# DATA 266 Lab 1 — Task 2: Yelp Polarity Sentiment Classification (member: manjot)

Three models trained **from scratch** — no pretrained embeddings, no pretrained language models:

| Role | Model | What changes vs. the others |
|---|---|---|
| Baseline | **fastText-style bag of n-grams** — averaged learned unigram + hashed-bigram embeddings → linear | order-free, very fast |
| Experimental 1 | **2-layer BiGRU + attention & max pooling** | adds recurrence / word order |
| Experimental 2 | **4-layer Transformer encoder (from scratch)** with a [CLS] token | self-attention instead of recurrence |

Differences from teammate Kavya's set (Mean-Embedding MLP, TextCNN, BiLSTM): different architectures for all three,
**full 560k training set** (vs. 200k subset), **lemmatisation** (vs. Porter stemming), contraction-aware **negation handling**,
an explicit **"!" token**, **head+tail truncation** (keeps the review's ending, where the verdict often is), and a held-out
validation split used for model selection (the test set is touched only once).

**Run modes:** `T2_MODE=smoke` (tiny, ~5 min on a laptop) or `T2_MODE=full` (GPU). Default `auto` = smoke unless CUDA is available.

In [ ]:
import os, re, sys, json, time, math, html, random, platform, resource, copy
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, confusion_matrix,
                             roc_auc_score, average_precision_score, matthews_corrcoef,
                             brier_score_loss, roc_curve, precision_recall_curve, f1_score)
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from scipy.stats import binomtest, chi2

SEED = 266
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"   # smoke runs use CPU (avoids MPS gaps for packed GRUs)
MODE = os.environ.get("T2_MODE", "auto").lower()
SMOKE = (device != "cuda") if MODE == "auto" else MODE == "smoke"

# ---- paths (notebook lives in task2_sentiment/manjot/src) ----
MEMBER_DIR = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
TASK_DIR = MEMBER_DIR.parent
REPO = TASK_DIR.parent
DATA_DIR = TASK_DIR / "data"
SFX = "_smoke" if SMOKE else ""
OUT = MEMBER_DIR / ("outputs/smoke" if SMOKE else "outputs")
CKPT = MEMBER_DIR / "checkpoints"
PROC = MEMBER_DIR / "data_processed"
MANIFEST = REPO / "reproducibility" / "manifests"
RAWLOG = REPO / "reproducibility" / "raw_logs"
for d in (DATA_DIR, OUT, CKPT, PROC, MANIFEST, RAWLOG):
    d.mkdir(parents=True, exist_ok=True)
RUN_ID = time.strftime("%Y%m%d_%H%M%S") + SFX

CFG = dict(
    max_train=None, max_test=None, val_frac=0.05, max_len=256, max_vocab=60_000, min_freq=3,
    batch=256, bootstrap=1000, ablation_rows=60_000,
    models={
        "baseline_fasttext_bigram": dict(epochs=5, lr=5e-3, wd=0.0, amp=True),
        "exp1_bigru_attention":     dict(epochs=4, lr=2e-3, wd=1e-2, amp=False),
        "exp2_transformer":         dict(epochs=5, lr=5e-4, wd=1e-2, amp=True),
    },
)
if SMOKE:
    CFG.update(max_train=4000, max_test=1000, max_len=128, max_vocab=5000, min_freq=2,
               batch=64, bootstrap=100, ablation_rows=3000)
    for m in CFG["models"].values():
        m["epochs"] = 1

print(f"device={device} | SMOKE={SMOKE} | run_id={RUN_ID}")
print(json.dumps({k: v for k, v in CFG.items() if k != "models"}, indent=1))
print(json.dumps(CFG["models"], indent=1))

In [ ]:
# Hardware / software manifest (recorded for every run)
def hardware_info():
    info = dict(run_id=RUN_ID, device=device, python=platform.python_version(), torch=torch.__version__,
                platform=platform.platform(), cpu=platform.processor() or platform.machine(),
                cpu_count=os.cpu_count())
    if device == "cuda":
        p = torch.cuda.get_device_properties(0)
        info.update(gpu=p.name, gpu_mem_gb=round(p.total_memory / 1e9, 1), cuda=torch.version.cuda)
    elif device == "mps":
        info.update(gpu="Apple Silicon GPU (MPS)")
    return info

HW = hardware_info()
HW_NAME = HW.get("gpu", HW["cpu"])
(MANIFEST / f"manjot_task2_hardware{SFX}.json").write_text(json.dumps(HW, indent=1))
HW

## 1. Load Yelp Polarity
Hugging Face `fancyzhx/yelp_polarity`: 560,000 train / 38,000 test reviews, label 0 = negative (1–2 stars), 1 = positive (4–5 stars).
The raw data is cached in `task2_sentiment/data/` (git-ignored; zipped to Google Drive for the README).

In [ ]:
from datasets import load_dataset
ds = load_dataset("fancyzhx/yelp_polarity", cache_dir=str(DATA_DIR / "hf_cache"))
train_raw = ds["train"].to_pandas()[["text", "label"]]
test_raw = ds["test"].to_pandas()[["text", "label"]]
if CFG["max_train"]:
    train_raw = train_raw.sample(CFG["max_train"], random_state=SEED).reset_index(drop=True)
if CFG["max_test"]:
    test_raw = test_raw.sample(CFG["max_test"], random_state=SEED).reset_index(drop=True)
print("train", train_raw.shape, "| test", test_raw.shape)
train_raw.head(3)

## 2. Data analysis (2.1.1) and malformed-entry handling (2.1.2)

In [ ]:
def eda(df, name):
    t = df["text"]
    words = t.fillna("").str.split().str.len()
    return {
        "split": name, "rows": len(df),
        "missing_text": int(t.isna().sum()),
        "empty_or_whitespace": int(t.fillna("").str.strip().eq("").sum()),
        "exact_duplicates": int(t.duplicated().sum()),
        "pct_positive": round(float(df["label"].mean()) * 100, 2),
        "literal_backslash_n": int(t.str.contains("\\n", regex=False).sum()),
        "escaped_quotes": int(t.str.contains(r'\"', regex=False).sum()),
        "html_entities": int(t.str.contains(r"&[a-z]+;|&#\d+;", regex=True).sum()),
        "non_ascii_rows": int(t.str.contains(r"[^\x00-\x7f]", regex=True).sum()),
        **{f"words_p{q}": float(words.quantile(q / 100)) for q in (5, 50, 90, 95, 99)},
        "words_mean_neg": round(float(words[df.label == 0].mean()), 1),
        "words_mean_pos": round(float(words[df.label == 1].mean()), 1),
    }

eda_tbl = pd.DataFrame([eda(train_raw, "train"), eda(test_raw, "test")]).set_index("split")
overlap = len(set(train_raw.text) & set(test_raw.text))
print("train/test exact text overlap:", overlap)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for lab, name in [(0, "negative"), (1, "positive")]:
    w = train_raw.loc[train_raw.label == lab, "text"].str.split().str.len()
    ax[0].hist(w.clip(upper=1000), bins=60, alpha=0.6, label=name)
ax[0].set_xlabel("review length (words, clipped at 1000)"); ax[0].set_ylabel("count"); ax[0].legend()
ax[0].set_title("Length distribution by class (train)")
train_raw.label.value_counts().sort_index().plot.bar(ax=ax[1], color=["#c0504d", "#4f81bd"])
ax[1].set_xticklabels(["negative (0)", "positive (1)"], rotation=0); ax[1].set_title("Class balance (train)")
plt.tight_layout(); plt.savefig(OUT / "eda_length_class.png", dpi=120); plt.show()
eda_tbl.T

In [ ]:
# Handling: fill missing, drop empty rows, drop exact duplicate reviews in train,
# drop train reviews that also appear verbatim in test (leakage). The test set is never modified.
def clean_split(df, is_train):
    df = df.copy()
    df["text"] = df["text"].fillna("").astype(str)
    n0 = len(df)
    df = df[df["text"].str.strip() != ""]
    removed = {"empty": n0 - len(df)}
    if is_train:
        n1 = len(df); df = df.drop_duplicates("text"); removed["duplicates"] = n1 - len(df)
        n2 = len(df); df = df[~df["text"].isin(set(test_raw.text))]; removed["test_leakage"] = n2 - len(df)
    return df.reset_index(drop=True), removed

train_df, rm_train = clean_split(train_raw, True)
test_df, rm_test = clean_split(test_raw, False)
print("removed from train:", rm_train, "| removed from test:", rm_test)

# stratified validation split, used for early stopping / model selection (test stays untouched)
rng = np.random.default_rng(SEED)
val_idx = np.concatenate([rng.choice(np.where(train_df.label == c)[0],
                                     int(CFG["val_frac"] * (train_df.label == c).sum()), replace=False)
                          for c in (0, 1)])
val_mask = np.zeros(len(train_df), bool); val_mask[val_idx] = True
val_df = train_df[val_mask].reset_index(drop=True)
train_df = train_df[~val_mask].reset_index(drop=True)
print(f"train {len(train_df):,} | val {len(val_df):,} | test {len(test_df):,}")

## 3. Text preprocessing (2.1.3) and tokenisation (2.1.4)
Pipeline (applied identically to every split):
1. **Fix malformed escapes** — Yelp text contains literal `\n` and `\"`; HTML entities are unescaped.
2. **Lowercase**.
3. **Negation-aware contractions** — `don't → do not`, `can't → can not`, `won't → will not`, so the negation survives as its own token.
4. **Exclamation marks → `xxexcl` token** (they carry sentiment), then **remove all other punctuation / special characters**.
5. **Stop-word removal** (scikit-learn English list) but **keeping sentiment-bearing words**: negations, contrast words and intensifiers.
6. **Lemmatisation** with WordNet (verb, then noun) — keeps real words (`ordered → order`, `was → be`) unlike stemming (`servic`).
7. **Tokenise** on the cleaned whitespace-separated words; vocabulary = top 60k words with count ≥ 3; `<pad>`=0, `<unk>`=1.
8. **Head + tail truncation** to 256 tokens: the first 128 + last 128 tokens of long reviews (the final verdict is often at the end).

In [ ]:
import nltk
try:
    nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
    from nltk.stem import WordNetLemmatizer
    _wnl = WordNetLemmatizer(); _wnl.lemmatize("tests")
    LEMMA_OK = True
except Exception as e:                       # fallback keeps the notebook runnable offline
    from nltk.stem import PorterStemmer
    _ps = PorterStemmer(); LEMMA_OK = False
    print("WordNet unavailable -> Porter stemming fallback:", e)
from nltk.stem import PorterStemmer
_porter = PorterStemmer()

KEEP = {"no", "not", "nor", "never", "none", "nothing", "nobody", "nowhere", "neither", "cannot", "without",
        "but", "however", "although", "though", "yet", "very", "too", "few", "less", "least", "more", "most",
        "much", "only", "against", "again", "well", "enough"}
STOP = (set(ENGLISH_STOP_WORDS) | {"ll", "ve", "re", "s", "d", "m", "t"}) - KEEP   # + contraction leftovers
_lemma_cache, _stem_cache = {}, {}

def lemma(w):
    r = _lemma_cache.get(w)
    if r is None:
        if LEMMA_OK:
            r = _wnl.lemmatize(w, "v")
            if r == w:
                r = _wnl.lemmatize(w, "n")
        else:
            r = _ps.stem(w)
        _lemma_cache[w] = r
    return r

def stem(w):
    r = _stem_cache.get(w)
    if r is None:
        r = _porter.stem(w); _stem_cache[w] = r
    return r

RE_ESC = re.compile(r'\\n|\\r|\\t|\\"')
RE_NT = [(re.compile(r"\bcan'?t\b"), "can not"), (re.compile(r"\bwon'?t\b"), "will not"),
         (re.compile(r"\bain'?t\b"), "is not"), (re.compile(r"n't\b"), " not")]
RE_EXCL = re.compile(r"!+")
RE_TOK = re.compile(r"[a-z0-9]+")

def preprocess(text, variant="final"):
    """variant: basic (lower+punct) | stop | final (stop+lemma) | stem (stop+porter)"""
    t = html.unescape(RE_ESC.sub(" ", text)).lower()
    for rx, rep in RE_NT:
        t = rx.sub(rep, t)
    t = RE_EXCL.sub(" xxexcl ", t)
    toks = RE_TOK.findall(t)
    if variant == "basic":
        return toks
    toks = [w for w in toks if w not in STOP]
    if variant == "stop":
        return toks
    if variant == "stem":
        return [stem(w) for w in toks]
    return [lemma(w) for w in toks]

ex = "The food wasn't great!!! Service was SLOW,\\n but the staff were very friendly &amp; we'll be back."
for v in ("basic", "stop", "final", "stem"):
    print(f"{v:6s} -> {preprocess(ex, v)}")

In [ ]:
def tokenize_all(texts, variant="final"):
    texts = list(texts)
    if sys.platform.startswith("linux") and len(texts) > 20_000:     # fork-based parallelism on the GPU box
        import multiprocessing as mp
        from functools import partial
        with mp.get_context("fork").Pool(min(16, os.cpu_count() or 2)) as pool:
            return pool.map(partial(preprocess, variant=variant), texts, chunksize=2000)
    return [preprocess(t, variant) for t in texts]

t0 = time.time()
tok_train = tokenize_all(train_df.text); tok_val = tokenize_all(val_df.text); tok_test = tokenize_all(test_df.text)
print(f"tokenised {len(tok_train) + len(tok_val) + len(tok_test):,} reviews in {time.time() - t0:.0f}s")

counter = Counter(w for toks in tok_train for w in toks)
itos = ["<pad>", "<unk>"] + [w for w, c in counter.most_common(CFG["max_vocab"] - 2) if c >= CFG["min_freq"]]
stoi = {w: i for i, w in enumerate(itos)}
V = len(itos)
lens = np.array([len(t) for t in tok_train])
oov = np.mean([w not in stoi for t in tok_test[:5000] for w in t])
prep_summary = dict(vocab_size=V, distinct_train_tokens=len(counter), lemmatizer="wordnet" if LEMMA_OK else "porter",
                    tokens_p50=float(np.percentile(lens, 50)), tokens_p95=float(np.percentile(lens, 95)),
                    tokens_p99=float(np.percentile(lens, 99)), pct_truncated=float((lens > CFG["max_len"]).mean() * 100),
                    test_oov_rate_pct=float(oov * 100), removed_train=rm_train, removed_test=rm_test)
(PROC / f"preprocessing_summary{SFX}.json").write_text(json.dumps(prep_summary, indent=1))
(PROC / f"vocab{SFX}.json").write_text(json.dumps(itos[:CFG["max_vocab"]]))
print(json.dumps(prep_summary, indent=1))
print("most common:", counter.most_common(25))

In [ ]:
L = CFG["max_len"]
def encode(tok_lists):
    X = np.zeros((len(tok_lists), L), dtype=np.int32)
    for i, toks in enumerate(tok_lists):
        ids = [stoi.get(w, 1) for w in toks]
        if len(ids) > L:                                   # head + tail truncation
            ids = ids[: L // 2] + ids[-(L - L // 2):]
        X[i, :len(ids)] = ids
    return X

Xtr, Xva, Xte = encode(tok_train), encode(tok_val), encode(tok_test)
ytr, yva, yte = (train_df.label.values.astype(np.int64), val_df.label.values.astype(np.int64),
                 test_df.label.values.astype(np.int64))
print("encoded:", Xtr.shape, Xva.shape, Xte.shape, "| example ids:", Xtr[0, :12])

### Preprocessing ablation (evidence for the pipeline choice)
Same fastText baseline, same subset, one epoch, four preprocessing variants; scored on a held-out slice.

In [ ]:
def quick_fasttext_score(variant, n=CFG["ablation_rows"]):
    sub = train_df.iloc[:n]
    split = int(0.8 * len(sub))
    toks = tokenize_all(sub.text, variant)
    cnt = Counter(w for t in toks[:split] for w in t)
    vocab = {w: i + 2 for i, (w, c) in enumerate(cnt.most_common(30_000)) if c >= 2}
    def enc(ts):
        X = np.zeros((len(ts), L), dtype=np.int64)
        for i, t in enumerate(ts):
            ids = [vocab.get(w, 1) for w in t]
            ids = ids[: L // 2] + ids[-(L - L // 2):] if len(ids) > L else ids
            X[i, :len(ids)] = ids
        return torch.from_numpy(X)
    Xa, Xb = enc(toks[:split]), enc(toks[split:])
    ya, yb = torch.tensor(sub.label.values[:split]), sub.label.values[split:]
    torch.manual_seed(SEED)
    m = FastTextBigram(len(vocab) + 2, dim=64, buckets=50_000).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=5e-3, weight_decay=0)
    m.train()
    for i in range(0, len(Xa), 256):
        xb, yb_ = Xa[i:i + 256].to(device), ya[i:i + 256].to(device)
        opt.zero_grad(); F.cross_entropy(m(xb), yb_).backward(); opt.step()
    m.eval(); preds = []
    with torch.no_grad():
        for i in range(0, len(Xb), 1024):
            preds.append(m(Xb[i:i + 1024].to(device)).argmax(1).cpu().numpy())
    p = np.concatenate(preds)
    return dict(variant=variant, vocab=len(vocab) + 2, accuracy=accuracy_score(yb, p),
                macro_f1=f1_score(yb, p, average="macro"))

## 4. Models (2.2.1–2.2.2)
* **Baseline — fastText-style bag of n-grams.** Learned 100-d embeddings for every unigram *and* every adjacent word pair
  (bigrams hashed into 200k buckets, so "not good" gets its own vector), averaged, then one linear layer.
  Justification: the standard strong, cheap baseline for review polarity; it shows how far word-presence + local negation gets.
* **Experimental 1 — 2-layer BiGRU with attention + max pooling.** 128-d embeddings, 128 hidden units per direction,
  packed sequences (padding ignored), additive attention pooling concatenated with max pooling.
  Justification: models word order and long-range context ("I wanted to love this place, but…"); attention lets it focus on the verdict.
* **Experimental 2 — 4-layer Transformer encoder from scratch.** 128-d tokens + learned positions, 4 heads, FFN 512, pre-norm,
  [CLS] token for classification. Justification: self-attention relates any two words directly regardless of distance — the
  architecture behind modern NLP, here trained only on Yelp text.

In [ ]:
class FastTextBigram(nn.Module):
    def __init__(self, vocab, dim=100, buckets=200_000, dropout=0.1):
        super().__init__()
        self.V, self.B = vocab, buckets
        self.emb = nn.EmbeddingBag(vocab + buckets, dim, mode="mean", padding_idx=0)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(dim, 2)
    def forward(self, x):
        a, b = x[:, :-1], x[:, 1:]
        bi = (a * 1_000_003 + b) % self.B + self.V              # hashed bigram ids
        bi = torch.where((a > 0) & (b > 0), bi, torch.zeros_like(bi))
        return self.fc(self.drop(self.emb(torch.cat([x, bi], 1))))

class BiGRUAttention(nn.Module):
    def __init__(self, vocab, dim=128, hidden=128, layers=2, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=0)
        self.emb_drop = nn.Dropout(0.2)
        self.gru = nn.GRU(dim, hidden, layers, batch_first=True, bidirectional=True, dropout=dropout)
        self.att = nn.Linear(2 * hidden, 1)
        self.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(4 * hidden, 2))
    def forward(self, x):
        mask = x != 0
        lengths = mask.sum(1).clamp(min=1).cpu()
        e = self.emb_drop(self.emb(x))
        packed = nn.utils.rnn.pack_padded_sequence(e, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.gru(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        scores = self.att(out).squeeze(-1).masked_fill(~mask, -1e4)
        attn = (torch.softmax(scores, 1).unsqueeze(-1) * out).sum(1)
        mx = out.masked_fill(~mask.unsqueeze(-1), -1e4).amax(1)
        return self.fc(torch.cat([attn, mx], 1))

class TransformerClassifier(nn.Module):
    def __init__(self, vocab, max_len, dim=128, heads=4, layers=4, ff=512, dropout=0.1):
        super().__init__()
        self.tok = nn.Embedding(vocab, dim, padding_idx=0)
        self.pos = nn.Embedding(max_len + 1, dim)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        layer = nn.TransformerEncoderLayer(dim, heads, ff, dropout, batch_first=True, norm_first=True, activation="gelu")
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)
        self.fc = nn.Linear(dim, 2)
    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T + 1, device=x.device)
        h = torch.cat([self.cls.expand(B, -1, -1), self.tok(x)], 1) + self.pos(pos)
        pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=x.device), x == 0], 1)
        h = self.enc(h, src_key_padding_mask=pad)
        return self.fc(self.norm(h[:, 0]))

def build(name):
    torch.manual_seed(SEED)
    if name.startswith("baseline"):
        return FastTextBigram(V, buckets=20_000 if SMOKE else 200_000)
    if name.startswith("exp1"):
        return BiGRUAttention(V)
    return TransformerClassifier(V, L)

for n in CFG["models"]:
    print(f"{n:28s} {sum(p.numel() for p in build(n).parameters()):>12,} parameters")

In [ ]:
abl = pd.DataFrame([quick_fasttext_score(v) for v in ("basic", "stop", "stem", "final")])
abl.to_csv(MEMBER_DIR / f"preprocessing_ablation{SFX}.csv", index=False)
abl

## 5. Training (validation-based model selection)
AdamW + one-cycle LR schedule, gradient clipping 1.0, bf16 autocast on GPU for the baseline and Transformer
(the GRU stays fp32 for cuDNN stability). After every epoch the model is scored on the validation split; the best-macro-F1
epoch is kept. Throughput, wall time and peak memory are recorded per model.

In [ ]:
def batches(X, y, bs, shuffle):
    idx = np.random.permutation(len(X)) if shuffle else np.arange(len(X))
    for i in range(0, len(X), bs):
        j = idx[i:i + bs]
        yield torch.from_numpy(X[j]).long(), torch.from_numpy(y[j])

def autocast(on):
    if on and device == "cuda":
        return torch.autocast("cuda", dtype=torch.bfloat16)
    return torch.autocast("cpu", enabled=False)

@torch.no_grad()
def predict(model, X, amp, bs=1024):
    model.eval(); probs = []
    t0 = time.time()
    for xb, _ in batches(X, np.zeros(len(X), np.int64), bs, False):
        with autocast(amp):
            logits = model(xb.to(device))
        probs.append(torch.softmax(logits.float(), 1)[:, 1].cpu().numpy())
    if device == "cuda": torch.cuda.synchronize()
    return np.concatenate(probs), len(X) / (time.time() - t0)

def peak_mem_mb():
    if device == "cuda":
        return torch.cuda.max_memory_allocated() / 2**20
    if device == "mps":
        return torch.mps.driver_allocated_memory() / 2**20
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return r / 2**20 if sys.platform == "darwin" else r / 1024

def train(name, spec):
    model = build(name).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=spec["lr"], weight_decay=spec["wd"])
    steps = spec["epochs"] * math.ceil(len(Xtr) / CFG["batch"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=spec["lr"], total_steps=steps, pct_start=0.1)
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    hist, best, best_f1 = [], None, -1
    t_start = time.time()
    for ep in range(1, spec["epochs"] + 1):
        model.train(); t0 = time.time(); tot, n = 0.0, 0
        for xb, yb in batches(Xtr, ytr, CFG["batch"], True):
            xb, yb = xb.to(device), yb.to(device)
            with autocast(spec["amp"]):
                loss = F.cross_entropy(model(xb).float(), yb)
            opt.zero_grad(set_to_none=True); loss.backward()
            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            tot += loss.item() * len(yb); n += len(yb)
        ep_time = time.time() - t0
        pv, _ = predict(model, Xva, spec["amp"])
        val_f1 = f1_score(yva, (pv >= 0.5).astype(int), average="macro")
        val_loss = float(-np.mean(yva * np.log(pv + 1e-7) + (1 - yva) * np.log(1 - pv + 1e-7)))
        hist.append(dict(model=name, epoch=ep, train_loss=tot / n, val_loss=val_loss, val_macro_f1=val_f1,
                         val_acc=accuracy_score(yva, pv >= 0.5), train_ex_per_s=n / ep_time,
                         epoch_s=ep_time, last_grad_norm=float(gn), nan_loss=int(not np.isfinite(tot))))
        print(f"{name} | epoch {ep}/{spec['epochs']} | train_loss {tot / n:.4f} | val_loss {val_loss:.4f} | "
              f"val_macroF1 {val_f1:.4f} | {n / ep_time:,.0f} ex/s", flush=True)
        if val_f1 > best_f1:
            best_f1, best = val_f1, copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
    train_time = time.time() - t_start
    model.load_state_dict(best)
    torch.save(best, CKPT / f"{name}{SFX}.pt")
    pd.DataFrame(hist).to_csv(OUT / f"training_history_{name}.csv", index=False)
    return model, hist, train_time, peak_mem_mb()

runs = {}
for name, spec in CFG["models"].items():
    model, hist, ttime, pmem = train(name, spec)
    probs, inf_rate = predict(model, Xte, spec["amp"])
    runs[name] = dict(model=model, hist=hist, train_time=ttime, peak_mem=pmem, probs=probs,
                      preds=(probs >= 0.5).astype(int), infer_ex_per_s=inf_rate,
                      train_ex_per_s=float(np.mean([h["train_ex_per_s"] for h in hist])),
                      best_epoch=int(np.argmax([h["val_macro_f1"] for h in hist]) + 1))
    print(f"   -> {name}: best epoch {runs[name]['best_epoch']}, train {ttime:.0f}s, peak mem {pmem:.0f} MB, "
          f"test acc {accuracy_score(yte, runs[name]['preds']):.4f}\n", flush=True)
pd.concat([pd.DataFrame(r["hist"]) for r in runs.values()]).to_csv(RAWLOG / f"manjot_task2_{RUN_ID}.csv", index=False)

## 6. Evaluation — full metrics list (2.2.3)

In [ ]:
def ece_score(y, p, bins=15):
    conf = np.maximum(p, 1 - p); pred = (p >= 0.5).astype(int); correct = (pred == y).astype(float)
    edges = np.linspace(0.5, 1.0, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi) if lo > 0.5 else (conf >= lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def _bin_stats(y, p):        # vectorised over bootstrap rows: y, p are [n_boot, n]
    tp = ((p == 1) & (y == 1)).sum(1).astype(float); tn = ((p == 0) & (y == 0)).sum(1).astype(float)
    fp = ((p == 1) & (y == 0)).sum(1).astype(float); fn = ((p == 0) & (y == 1)).sum(1).astype(float)
    acc = (tp + tn) / y.shape[1]
    f1p = 2 * tp / np.maximum(2 * tp + fp + fn, 1); f1n = 2 * tn / np.maximum(2 * tn + fn + fp, 1)
    mcc = (tp * tn - fp * fn) / np.sqrt(np.maximum((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn), 1e-12))
    return acc, (f1p + f1n) / 2, mcc

def bootstrap_ci(y, p, n_boot=CFG["bootstrap"], chunk=100):
    rng = np.random.default_rng(SEED); out = []
    for s in range(0, n_boot, chunk):
        idx = rng.integers(0, len(y), (min(chunk, n_boot - s), len(y)))
        out.append(np.stack(_bin_stats(y[idx], p[idx]), 1))
    out = np.concatenate(out)
    return {k: (float(np.percentile(out[:, i], 2.5)), float(np.percentile(out[:, i], 97.5)))
            for i, k in enumerate(["accuracy", "macro_f1", "mcc"])}

def metrics_for(name, r):
    p, yhat = r["probs"], r["preds"]
    row = {"model": name, "accuracy": accuracy_score(yte, yhat)}
    for avg in ("macro", "micro", "weighted"):
        pr, rc, f1, _ = precision_recall_fscore_support(yte, yhat, average=avg, zero_division=0)
        row.update({f"precision_{avg}": pr, f"recall_{avg}": rc, f"f1_{avg}": f1})
    tn, fp, fn, tp = confusion_matrix(yte, yhat, labels=[0, 1]).ravel()
    row.update(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
               roc_auc=roc_auc_score(yte, p), pr_auc=average_precision_score(yte, p),
               mcc=matthews_corrcoef(yte, yhat), brier=brier_score_loss(yte, p), ece=ece_score(yte, p))
    for k, (lo, hi) in bootstrap_ci(yte, yhat).items():
        row[f"{k}_ci95_low"], row[f"{k}_ci95_high"] = lo, hi
    row.update(params=sum(q.numel() for q in r["model"].parameters()), train_time_s=r["train_time"],
               train_ex_per_s=r["train_ex_per_s"], infer_ex_per_s=r["infer_ex_per_s"], peak_mem_mb=r["peak_mem"],
               best_epoch=r["best_epoch"], epochs=CFG["models"][name]["epochs"], hardware=HW_NAME,
               train_rows=len(Xtr), test_rows=len(Xte))
    return row

metrics = pd.DataFrame([metrics_for(n, r) for n, r in runs.items()])
metrics.to_csv(MEMBER_DIR / f"metrics_report{SFX}.csv", index=False)
show = ["model", "accuracy", "f1_macro", "f1_micro", "f1_weighted", "precision_macro", "recall_macro", "roc_auc", "pr_auc",
        "mcc", "brier", "ece", "params", "train_time_s", "train_ex_per_s", "infer_ex_per_s", "peak_mem_mb"]
metrics[show].round(4)

In [ ]:
ci_cols = [c for c in metrics.columns if "ci95" in c]
metrics[["model"] + ci_cols].round(4)

In [ ]:
# Confusion matrices, ROC, precision-recall and reliability (calibration) curves
names = list(runs)
fig, ax = plt.subplots(4, len(names), figsize=(4.6 * len(names), 16))
ax = np.array(ax).reshape(4, len(names))
for j, n in enumerate(names):
    p, yhat = runs[n]["probs"], runs[n]["preds"]
    cm = confusion_matrix(yte, yhat, labels=[0, 1])
    ax[0, j].imshow(cm, cmap="Blues")
    for (a, b), v in np.ndenumerate(cm):
        ax[0, j].text(b, a, f"{v:,}", ha="center", va="center", color="white" if v > cm.max() / 2 else "black")
    ax[0, j].set_xticks([0, 1], ["neg", "pos"]); ax[0, j].set_yticks([0, 1], ["neg", "pos"])
    ax[0, j].set_xlabel("predicted"); ax[0, j].set_ylabel("true"); ax[0, j].set_title(n, fontsize=10)
    fpr, tpr, _ = roc_curve(yte, p); ax[1, j].plot(fpr, tpr); ax[1, j].plot([0, 1], [0, 1], "k--", lw=0.8)
    ax[1, j].set_title(f"ROC (AUC {roc_auc_score(yte, p):.4f})"); ax[1, j].set_xlabel("FPR"); ax[1, j].set_ylabel("TPR")
    pr, rc, _ = precision_recall_curve(yte, p); ax[2, j].plot(rc, pr)
    ax[2, j].set_title(f"PR (AP {average_precision_score(yte, p):.4f})"); ax[2, j].set_xlabel("recall"); ax[2, j].set_ylabel("precision")
    conf = np.maximum(p, 1 - p); correct = ((p >= 0.5) == yte)
    edges = np.linspace(0.5, 1, 11); mids, accs = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf <= hi)
        if m.sum() > 20:
            mids.append(conf[m].mean()); accs.append(correct[m].mean())
    ax[3, j].plot([0.5, 1], [0.5, 1], "k--", lw=0.8); ax[3, j].plot(mids, accs, "o-")
    ax[3, j].set_title(f"Reliability (ECE {ece_score(yte, p):.4f})"); ax[3, j].set_xlabel("confidence"); ax[3, j].set_ylabel("accuracy")
plt.tight_layout(); plt.savefig(OUT / "eval_curves_all_models.png", dpi=120); plt.show()
for n in names:
    pd.DataFrame(confusion_matrix(yte, runs[n]["preds"], labels=[0, 1]), index=["true_neg", "true_pos"],
                 columns=["pred_neg", "pred_pos"]).to_csv(OUT / f"confusion_{n}.csv")

In [ ]:
# Training curves
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for n, r in runs.items():
    h = pd.DataFrame(r["hist"])
    ax[0].plot(h.epoch, h.train_loss, "o-", label=n); ax[1].plot(h.epoch, h.val_loss, "o-", label=n)
    ax[2].plot(h.epoch, h.val_macro_f1, "o-", label=n)
for a, t in zip(ax, ["train loss", "validation loss", "validation macro-F1"]):
    a.set_title(t); a.set_xlabel("epoch")
ax[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(OUT / "training_curves.png", dpi=120); plt.show()

### Paired McNemar tests (baseline vs. each experimental model)
Only the reviews where the two models disagree matter: *b* = baseline right & other wrong, *c* = baseline wrong & other right.
Exact binomial test when b + c < 25, otherwise χ² with continuity correction.

In [ ]:
def mcnemar(y, pa, pb):
    a_ok, b_ok = pa == y, pb == y
    b, c = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    if b + c == 0:
        return dict(b=b, c=c, statistic=0.0, p_value=1.0, test="none")
    if b + c < 25:
        return dict(b=b, c=c, statistic=float(min(b, c)), p_value=binomtest(min(b, c), b + c, 0.5).pvalue, test="exact")
    stat = (abs(b - c) - 1) ** 2 / (b + c)
    return dict(b=b, c=c, statistic=stat, p_value=float(chi2.sf(stat, 1)), test="chi2_cc")

base = names[0]
mc = pd.DataFrame([{"comparison": f"{base} vs {n}", **mcnemar(yte, runs[base]["preds"], runs[n]["preds"]),
                    "acc_baseline": accuracy_score(yte, runs[base]["preds"]),
                    "acc_other": accuracy_score(yte, runs[n]["preds"])} for n in names[1:]])
mc["significant_at_0.05"] = mc.p_value < 0.05
mc.to_csv(MEMBER_DIR / f"mcnemar_tests{SFX}.csv", index=False)
mc

### Robustness: macro-F1 and error rate per data slice
Slices: review length (raw words: short < 50, medium 50–150, long > 150), contains a negation, contains a contrast word
("but / however / although"), and exclamation-heavy (≥ 3 "!").

In [ ]:
raw = test_df.text.str.lower()
nwords = test_df.text.str.split().str.len()
SLICES = {
    "len_short(<50w)": nwords < 50, "len_medium(50-150w)": (nwords >= 50) & (nwords <= 150), "len_long(>150w)": nwords > 150,
    "has_negation": raw.str.contains(r"\b(?:not|no|never|nothing|n't|cannot)\b|n't", regex=True),
    "has_contrast": raw.str.contains(r"\b(?:but|however|although|though|yet)\b", regex=True),
    "exclaim_heavy(>=3!)": test_df.text.str.count("!") >= 3,
    "all": pd.Series(True, index=test_df.index),
}
rows = []
for n, r in runs.items():
    for s, m in SLICES.items():
        m = m.values
        if m.sum() == 0:
            continue
        rows.append(dict(model=n, slice=s, n=int(m.sum()),
                         macro_f1=f1_score(yte[m], r["preds"][m], average="macro", zero_division=0),
                         error_rate=float((r["preds"][m] != yte[m]).mean())))
slices = pd.DataFrame(rows)
slices.to_csv(MEMBER_DIR / f"slice_metrics{SFX}.csv", index=False)
slices.pivot(index="slice", columns="model", values="macro_f1").round(4)

## 7. Error review — 20 cases from the best model (2.2.4)
5 confident false positives, 5 confident false negatives, 5 near-threshold errors, 5 failures from the weakest slice.
`suggested_type` is only a keyword hint — **`manual_error_type` and `testable_fix` must be filled in by hand** after reading each review.

In [ ]:
best = metrics.sort_values("f1_macro", ascending=False).iloc[0]["model"]
p, yhat = runs[best]["probs"], runs[best]["preds"]
err = pd.DataFrame({"text": test_df.text.values, "y_true": yte, "p_positive": p, "y_pred": yhat})
err["confidence"] = np.maximum(p, 1 - p); err["is_error"] = err.y_true != err.y_pred
err["n_words"] = nwords.values
sl = slices[(slices.model == best) & (slices.slice != "all") & (slices.n >= (50 if SMOKE else 300))]
if sl.empty:
    sl = slices[(slices.model == best) & (slices.slice != "all")]
worst_slice = sl.sort_values("macro_f1").iloc[0]["slice"]

pick = []
fp_ = err[(err.y_true == 0) & (err.y_pred == 1)].nlargest(5, "confidence").assign(group="confident_false_positive")
fn_ = err[(err.y_true == 1) & (err.y_pred == 0)].nlargest(5, "confidence").assign(group="confident_false_negative")
used = set(fp_.index) | set(fn_.index)
near = err[err.is_error & ~err.index.isin(used)].nsmallest(5, "confidence").assign(group="near_threshold")
used |= set(near.index)
sl_err = err[err.is_error & SLICES[worst_slice].values & ~err.index.isin(used)].nlargest(5, "confidence")
sl_err = sl_err.assign(group=f"slice_failure:{worst_slice}")

def suggest(t):
    t = t.lower(); tags = []
    if re.search(r"\bbut\b|however|although|though", t): tags.append("mixed/contrast")
    if re.search(r"n't|\bnot\b|\bnever\b|\bno\b", t): tags.append("negation")
    if re.search(r"\b(?:yeah right|thanks a lot|great job|so much for|lol)\b|\?\!", t): tags.append("possible sarcasm")
    if len(t.split()) > 300: tags.append("long review (truncated middle)")
    if re.search(r"\b(?:[1-5]) stars?\b", t): tags.append("explicit star mention")
    return "; ".join(tags) or "other / unclear"

review = pd.concat([fp_, fn_, near, sl_err]).reset_index(names="test_index")
review["suggested_type"] = review.text.map(suggest)
review["manual_error_type"] = ""; review["testable_fix"] = ""
review = review[["group", "test_index", "y_true", "y_pred", "p_positive", "n_words", "suggested_type",
                 "manual_error_type", "testable_fix", "text"]]
review.to_csv(MEMBER_DIR / f"error_review_candidates{SFX}.csv", index=False)
print("model reviewed:", best, "| weakest slice:", worst_slice)
pd.set_option("display.max_colwidth", 160)
review.drop(columns="text").assign(text=review.text.str[:160])

## 8. Comparison with teammate (2.3) and run summary

In [ ]:
mate = TASK_DIR / "Kavya" / "metrics_report.csv"
cols = ["model", "accuracy", "f1_macro", "roc_auc", "pr_auc", "mcc", "brier", "params", "train_time_s", "hardware"]
mine = metrics.assign(member="manjot")[["member"] + [c for c in cols if c in metrics.columns]]
if mate.exists():
    k = pd.read_csv(mate).rename(columns={"brier_score": "brier", "parameter_count": "params",
                                          "training_time_seconds": "train_time_s"})
    k = k.assign(member="Kavya")[["member"] + [c for c in cols if c in k.columns]]
    team = pd.concat([mine, k], ignore_index=True)
    team.to_csv(TASK_DIR / f"team_comparison{SFX}.csv", index=False)
    display(team.round(4))
else:
    print("Teammate metrics not found at", mate, "- showing mine only")
    display(mine.round(4))

summary = dict(run_id=RUN_ID, smoke=SMOKE, hardware=HW, config=CFG, preprocessing=prep_summary,
               best_model=best, weakest_slice=worst_slice,
               test_accuracy={n: float(accuracy_score(yte, r["preds"])) for n, r in runs.items()})
(RAWLOG / f"manjot_task2_{RUN_ID}_summary.json").write_text(json.dumps(summary, indent=1, default=str))
(MEMBER_DIR / f"config{SFX}.json").write_text(json.dumps(CFG, indent=1))
print("done:", RUN_ID)